# VoiceHack2026 — Problem 1: Call Quality Auto-Flagger

## Cell 1 — Install dependencies

In [ ]:
!pip install xgboost scikit-learn imbalanced-learn pandas numpy matplotlib seaborn shap sentence-transformers -q

## Cell 2 — Imports

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import f1_score, recall_score, precision_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import shap

import matplotlib.pyplot as plt
import seaborn as sns

print('All imports successful.')
print(f'XGBoost version: {xgb.__version__}')

## Cell 3 — Upload CSV files

In [ ]:
train_df = pd.read_csv('hackathon_train.csv')
val_df   = pd.read_csv('hackathon_val.csv')
test_df  = pd.read_csv('hackathon_test.csv')

print(f'Train : {train_df.shape}  |  Tickets: {train_df["has_ticket"].sum()}')
print(f'Val   : {val_df.shape}   |  Tickets: {val_df["has_ticket"].sum()}')
print(f'Test  : {test_df.shape}  |  Labels hidden')

## Cell 4 — EDA

In [ ]:
print('=== Column overview ===')
print(train_df.dtypes)
print('\n=== Missing values ===')
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Outcome distribution by ticket
outcome_ticket = train_df.groupby(['outcome', 'has_ticket']).size().unstack(fill_value=0)
outcome_ticket.plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'])
axes[0].set_title('Outcome vs Ticket')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

# Call duration distribution
train_df.groupby('has_ticket')['call_duration'].plot(
    kind='hist', bins=40, alpha=0.6, ax=axes[1], legend=True
)
axes[1].set_title('Call duration by ticket status')
axes[1].set_xlabel('Duration (seconds)')

# Response completeness
train_df.groupby('has_ticket')['response_completeness'].plot(
    kind='hist', bins=20, alpha=0.6, ax=axes[2], legend=True
)
axes[2].set_title('Response completeness by ticket status')

plt.tight_layout()
plt.show()

print('\nTicket rate by outcome:')
print(train_df.groupby('outcome')['has_ticket'].agg(['sum','count','mean']).rename(
    columns={'sum':'tickets','count':'total','mean':'rate'}
).sort_values('rate', ascending=False))

## Cell 5 — Feature Engineering helpers

In [ ]:
# ─── Keyword patterns ────────────────────────────────────────────────────────
OPT_OUT_PATTERNS = [
    r"not interested", r"don'?t (want|need)", r"please (stop|remove)",
    r"take me off", r"no thank you", r"no thanks", r"i'?m (good|fine|all set)"
]
WRONG_NUMBER_PATTERNS = [
    r"wrong number", r"no .{0,15} here", r"don'?t know (who|a)",
    r"wrong (person|house|address)"
]
MEDICAL_ADVICE_PATTERNS = [
    r"i (recommend|suggest|advise)", r"you should (take|try|use|reduce|increase)",
    r"the (dosage|dose) (is|should)", r"try (reducing|increasing|stopping)",
    r"(take|use) (more|less|extra|fewer)"
]
ESCALATION_PATTERNS = [
    r"chest pain", r"difficult.{0,10}breath", r"allergic reaction",
    r"emergency", r"call 911", r"severe"
]

def count_pattern_matches(text, patterns):
    """Count how many patterns match in a given text."""
    if not isinstance(text, str):
        return 0
    text_lower = text.lower()
    return sum(1 for p in patterns if re.search(p, text_lower))

def any_pattern_match(text, patterns):
    return int(count_pattern_matches(text, patterns) > 0)


# ─── Q&A response parser ─────────────────────────────────────────────────────
def parse_responses(responses_json):
    """Parse the responses_json column into a list of dicts."""
    if isinstance(responses_json, str):
        try:
            return json.loads(responses_json)
        except:
            return []
    if isinstance(responses_json, list):
        return responses_json
    return []


NUMERIC_QUESTIONS = [
    "current weight", "height", "weight have you lost", "goal weight"
]

def extract_numeric_answer(answer):
    """Extract first numeric value from an answer string."""
    if not isinstance(answer, str):
        return None
    nums = re.findall(r'\d+\.?\d*', answer)
    return float(nums[0]) if nums else None


def is_suspicious_weight(val):
    """Flag implausibly low weights that might be STT mishearings."""
    if val is None:
        return 0
    # Adults on weight-loss meds: plausible range 100-500 lbs
    return int(val < 80 or val > 550)


print('Helper functions defined.')

## Cell 6 — Full feature engineering function

In [ ]:
def engineer_features(df):
    """
    Build all 5 feature groups and return a new feature DataFrame.
    Input: raw dataframe with all columns
    Output: feature dataframe (no raw text columns)
    """
    feat = pd.DataFrame(index=df.index)

    # ── GROUP 1: Raw metadata features ────────────────────────────────────────
    feat['call_duration']          = df['call_duration'].fillna(0)
    feat['attempt_number']         = df['attempt_number'].fillna(1)
    feat['whisper_mismatch_count'] = df['whisper_mismatch_count'].fillna(0)
    feat['response_completeness']  = df['response_completeness'].fillna(0)
    feat['question_count']         = df['question_count'].fillna(14)
    feat['answered_count']         = df['answered_count'].fillna(0)
    feat['turn_count']             = df['turn_count'].fillna(0)
    feat['user_turn_count']        = df['user_turn_count'].fillna(0)
    feat['agent_turn_count']       = df['agent_turn_count'].fillna(0)
    feat['user_word_count']        = df['user_word_count'].fillna(0)
    feat['agent_word_count']       = df['agent_word_count'].fillna(0)
    feat['avg_user_turn_words']    = df['avg_user_turn_words'].fillna(0)
    feat['avg_agent_turn_words']   = df['avg_agent_turn_words'].fillna(0)
    feat['interruption_count']     = df['interruption_count'].fillna(0)
    feat['form_submitted']         = df['form_submitted'].astype(int)
    feat['hour_of_day']            = df['hour_of_day'].fillna(12)

    # Cyclical encoding for hour and day
    feat['hour_sin'] = np.sin(2 * np.pi * feat['hour_of_day'] / 24)
    feat['hour_cos'] = np.cos(2 * np.pi * feat['hour_of_day'] / 24)

    day_map = {'Monday':0,'Tuesday':1,'Wednesday':2,'Thursday':3,
               'Friday':4,'Saturday':5,'Sunday':6}
    day_num = df['day_of_week'].map(day_map).fillna(0)
    feat['dow_sin'] = np.sin(2 * np.pi * day_num / 7)
    feat['dow_cos'] = np.cos(2 * np.pi * day_num / 7)

    # Outcome one-hot
    outcomes = ['completed','incomplete','opted_out','scheduled',
                'escalated','wrong_number','voicemail']
    for o in outcomes:
        feat[f'outcome_{o}'] = (df['outcome'] == o).astype(int)

    # ── GROUP 2: Structural anomaly features ──────────────────────────────────
    # Completed but low response completeness
    feat['completed_low_completeness'] = (
        (df['outcome'] == 'completed') & (df['response_completeness'] < 0.8)
    ).astype(int)

    # Completed but very short call duration
    feat['completed_short_call'] = (
        (df['outcome'] == 'completed') & (df['call_duration'] < 120)
    ).astype(int)

    # Completed but form not submitted
    feat['completed_no_form'] = (
        (df['outcome'] == 'completed') & (~df['form_submitted'].astype(bool))
    ).astype(int)

    # High mismatch with completed outcome
    feat['high_mismatch_completed'] = (
        (df['whisper_mismatch_count'] >= 2) & (df['outcome'] == 'completed')
    ).astype(int)

    # Any high mismatch
    feat['any_high_mismatch'] = (df['whisper_mismatch_count'] >= 3).astype(int)

    # Duration per question answered (implausible pace)
    feat['duration_per_answer'] = df['call_duration'] / (df['answered_count'].replace(0, np.nan))
    feat['duration_per_answer'] = feat['duration_per_answer'].fillna(df['call_duration'])

    # Agent word ratio (agent dominated = possible skipping)
    total_words = df['user_word_count'] + df['agent_word_count']
    feat['agent_word_ratio'] = df['agent_word_count'] / total_words.replace(0, np.nan)
    feat['agent_word_ratio'] = feat['agent_word_ratio'].fillna(0.5)

    # Questions gap: expected 14 but fewer answered for completed calls
    feat['questions_gap'] = df['question_count'] - df['answered_count']
    feat['questions_gap_completed'] = feat['questions_gap'] * feat['outcome_completed']

    # ── GROUP 3: Q&A integrity features ───────────────────────────────────────
    responses_list = df['responses_json'].apply(parse_responses)

    def qa_features(responses):
        empty_count = 0
        suspicious_weight = 0
        answered = 0
        weight_val = None

        for r in responses:
            q = str(r.get('question', '')).lower()
            a = str(r.get('answer', ''))
            if a.strip() == '':
                empty_count += 1
            else:
                answered += 1
            if 'current weight' in q:
                weight_val = extract_numeric_answer(a)
                suspicious_weight = is_suspicious_weight(weight_val)

        return pd.Series({
            'qa_empty_answer_count':  empty_count,
            'qa_answered_count':      answered,
            'qa_suspicious_weight':   suspicious_weight,
            'qa_weight_value':        weight_val if weight_val else 0
        })

    qa_feats = responses_list.apply(qa_features)
    feat = pd.concat([feat, qa_feats], axis=1)

    # Discrepancy between metadata answered_count and actual non-empty responses
    feat['answered_count_discrepancy'] = abs(
        df['answered_count'] - feat['qa_answered_count']
    )

    # Fabricated answers: answered_count > 0 for a very short call
    feat['possible_fabrication'] = (
        (feat['qa_answered_count'] > feat['qa_answered_count'].mean()) &
        (df['call_duration'] < 100)
    ).astype(int)

    # ── GROUP 4: Transcript NLP features ─────────────────────────────────────
    transcript = df['transcript_text'].fillna('')

    feat['transcript_opt_out_signal']     = transcript.apply(
        lambda t: any_pattern_match(t, OPT_OUT_PATTERNS))
    feat['transcript_wrong_number_signal'] = transcript.apply(
        lambda t: any_pattern_match(t, WRONG_NUMBER_PATTERNS))
    feat['transcript_medical_advice']     = transcript.apply(
        lambda t: any_pattern_match(t, MEDICAL_ADVICE_PATTERNS))
    feat['transcript_escalation']         = transcript.apply(
        lambda t: any_pattern_match(t, ESCALATION_PATTERNS))

    # Number of agent questions asked (lines ending with ?)
    def count_agent_questions(t):
        agent_turns = re.findall(r'\[AGENT\]:.*?(?=\[|$)', t, re.DOTALL)
        return sum(1 for turn in agent_turns if '?' in turn)

    feat['agent_question_count'] = transcript.apply(count_agent_questions)

    # Gap between questions asked and answers recorded
    feat['question_answer_gap'] = feat['agent_question_count'] - df['answered_count'].fillna(0)
    feat['question_answer_gap'] = feat['question_answer_gap'].clip(lower=0)

    # Transcript length
    feat['transcript_word_count'] = transcript.apply(lambda t: len(t.split()))
    feat['transcript_char_count'] = transcript.apply(len)

    # Validation notes signals (post-call AI analysis)
    val_notes = df['validation_notes'].fillna('') if 'validation_notes' in df.columns else pd.Series([''] * len(df))
    feat['val_notes_has_mismatch']   = val_notes.str.contains('mismatch|MISMATCH', case=False, na=False).astype(int)
    feat['val_notes_has_fabricated'] = val_notes.str.contains('fabricat|invent', case=False, na=False).astype(int)
    feat['val_notes_has_skipped']    = val_notes.str.contains('skip|missed|never asked', case=False, na=False).astype(int)
    feat['val_notes_length']         = val_notes.apply(len)

    # ── GROUP 5: Cross-field sanity checks ────────────────────────────────────
    # Outcome contradictions
    feat['opted_out_wrong_outcome'] = (
        (df['outcome'] == 'wrong_number') &
        (feat['transcript_opt_out_signal'] == 1)
    ).astype(int)

    feat['opted_out_classified_completed'] = (
        (df['outcome'] == 'completed') &
        (feat['transcript_opt_out_signal'] == 1)
    ).astype(int)

    feat['wrong_number_misclassified'] = (
        (df['outcome'] == 'wrong_number') &
        (feat['transcript_opt_out_signal'] == 1)
    ).astype(int)

    # Whisper answer cross-check:
    # Check if the weight answer appears in whisper transcript
    whisper = df['whisper_transcript'].fillna('') if 'whisper_transcript' in df.columns else pd.Series([''] * len(df))

    def weight_in_whisper(row_idx):
        responses = responses_list.iloc[row_idx]
        wt = whisper.iloc[row_idx]
        for r in responses:
            q = str(r.get('question','')).lower()
            a = str(r.get('answer',''))
            if 'current weight' in q and a.strip():
                num = extract_numeric_answer(a)
                if num and str(int(num)) not in wt:
                    return 1
        return 0

    feat['weight_answer_not_in_whisper'] = [
        weight_in_whisper(i) for i in range(len(df))
    ]

    # Composite anomaly score (sum of all binary flags)
    binary_flags = [
        'completed_low_completeness', 'completed_short_call', 'completed_no_form',
        'high_mismatch_completed', 'any_high_mismatch',
        'qa_suspicious_weight', 'possible_fabrication',
        'transcript_opt_out_signal', 'transcript_wrong_number_signal',
        'transcript_medical_advice', 'transcript_escalation',
        'opted_out_wrong_outcome', 'opted_out_classified_completed',
        'val_notes_has_mismatch', 'val_notes_has_fabricated', 'val_notes_has_skipped',
        'weight_answer_not_in_whisper'
    ]
    feat['anomaly_score'] = feat[binary_flags].sum(axis=1)

    return feat.astype(float)


print('Feature engineering function defined.')

## Cell 7 — Build feature matrices

In [ ]:
print('Engineering features for train...')
X_train = engineer_features(train_df)
y_train = train_df['has_ticket'].astype(int)

print('Engineering features for val...')
X_val = engineer_features(val_df)
y_val = val_df['has_ticket'].astype(int)

print('Engineering features for test...')
X_test = engineer_features(test_df)

print(f'\nFeature matrix shape: {X_train.shape}')
print(f'Feature names ({len(X_train.columns)} total):')
print(list(X_train.columns))

## Cell 8 — Rule-based pre-filter

In [ ]:
def apply_rules(df, X_feat):
    """
    Deterministic rules that very reliably flag tickets.
    Returns a Series of 0/1 flags and a Series of rule names for diagnosis.
    """
    flags = pd.Series(0, index=df.index)
    reasons = pd.Series('', index=df.index)

    # Rule 1: Completed call but too few questions answered
    r1 = (df['outcome'] == 'completed') & (df['response_completeness'] < 0.7)
    flags[r1] = 1
    reasons[r1] = 'skipped_questions'

    # Rule 2: High STT mismatch
    r2 = df['whisper_mismatch_count'] >= 3
    flags[r2] = 1
    reasons[r2] += '|stt_mismatch'

    # Rule 3: Opted out but classified as wrong_number
    r3 = X_feat['opted_out_wrong_outcome'] == 1
    flags[r3] = 1
    reasons[r3] += '|wrong_number_misclassify'

    # Rule 4: Completed but opt-out signal in transcript
    r4 = X_feat['opted_out_classified_completed'] == 1
    flags[r4] = 1
    reasons[r4] += '|outcome_mismatch'

    # Rule 5: Medical advice given
    r5 = X_feat['transcript_medical_advice'] == 1
    flags[r5] = 1
    reasons[r5] += '|medical_advice'

    # Rule 6: Suspicious weight with completed outcome
    r6 = (X_feat['qa_suspicious_weight'] == 1) & (df['outcome'] == 'completed')
    flags[r6] = 1
    reasons[r6] += '|suspicious_weight_stt'

    # Rule 7: Validation notes flag fabrication or skipping
    r7 = (X_feat['val_notes_has_fabricated'] == 1) | (X_feat['val_notes_has_skipped'] == 1)
    flags[r7] = 1
    reasons[r7] += '|validation_flag'

    return flags, reasons


train_rule_flags, train_rule_reasons = apply_rules(train_df, X_train)
val_rule_flags, val_rule_reasons     = apply_rules(val_df, X_val)

print('Rule-based results on TRAIN set:')
print(classification_report(y_train, train_rule_flags, target_names=['no_ticket','ticket']))
print('\nRule-based results on VAL set:')
print(classification_report(y_val, val_rule_flags, target_names=['no_ticket','ticket']))

## Cell 9 — XGBoost classifier

In [ ]:
# Class imbalance ratio for scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
spw = neg_count / pos_count
print(f'Negative: {neg_count}, Positive: {pos_count}, scale_pos_weight: {spw:.2f}')

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=spw,
    eval_metric='aucpr',
    early_stopping_rounds=30,
    random_state=42,
    use_label_encoder=False
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

print(f'\nBest iteration: {xgb_model.best_iteration}')

## Cell 10 — Threshold tuning on validation set

In [ ]:
xgb_val_proba = xgb_model.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.05, 0.70, 0.01)
results = []
for t in thresholds:
    preds = (xgb_val_proba >= t).astype(int)
    f1  = f1_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    pre = precision_score(y_val, preds, zero_division=0)
    results.append({'threshold': t, 'f1': f1, 'recall': rec, 'precision': pre})

results_df = pd.DataFrame(results)
best_row = results_df.loc[results_df['f1'].idxmax()]
BEST_THRESHOLD = best_row['threshold']

print(f'Best threshold (max F1): {BEST_THRESHOLD:.2f}')
print(f'  F1={best_row["f1"]:.3f}  Recall={best_row["recall"]:.3f}  Precision={best_row["precision"]:.3f}')

# Plot threshold curves
plt.figure(figsize=(10, 4))
plt.plot(results_df['threshold'], results_df['f1'],        label='F1',        linewidth=2)
plt.plot(results_df['threshold'], results_df['recall'],    label='Recall',    linewidth=2)
plt.plot(results_df['threshold'], results_df['precision'], label='Precision', linewidth=2)
plt.axvline(BEST_THRESHOLD, color='red', linestyle='--', label=f'Best threshold = {BEST_THRESHOLD:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Threshold vs Metrics (Validation Set)')
plt.legend()
plt.tight_layout()
plt.show()

## Cell 11 — Weighted ensemble (rules + XGBoost)

In [ ]:
def ensemble_predict(rule_flags, xgb_proba, threshold=BEST_THRESHOLD,
                     rule_weight=0.35, xgb_weight=0.65):
    """
    Weighted soft-vote ensemble.
    rule_flags: binary 0/1 Series
    xgb_proba:  probability [0,1] array
    """
    combined = rule_weight * rule_flags.values + xgb_weight * xgb_proba
    return (combined >= threshold).astype(int), combined


# Evaluate on validation set
xgb_train_proba = xgb_model.predict_proba(X_train)[:, 1]
xgb_val_proba   = xgb_model.predict_proba(X_val)[:, 1]

val_preds, val_scores = ensemble_predict(val_rule_flags, xgb_val_proba)

print('=== ENSEMBLE — VALIDATION SET RESULTS ===')
print(classification_report(y_val, val_preds, target_names=['no_ticket','ticket']))

# Confusion matrix
cm = confusion_matrix(y_val, val_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['pred: no ticket','pred: ticket'],
            yticklabels=['actual: no ticket','actual: ticket'])
plt.title('Confusion Matrix — Validation Set')
plt.tight_layout()
plt.show()

## Cell 12 — SHAP feature importance

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_val)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val, max_display=20, show=False)
plt.title('SHAP Feature Importance (Validation Set)')
plt.tight_layout()
plt.show()

# Bar chart of mean absolute SHAP
mean_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_val.columns)
top20 = mean_shap.sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
top20.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 20 Features by Mean |SHAP|')
plt.xlabel('Mean |SHAP value|')
plt.tight_layout()
plt.show()

## Cell 13 — Cross-validation on Train+Val combined

In [ ]:
# Combine train + val for final model training
X_all = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_all = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_f1     = []
cv_recall = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_all, y_all)):
    Xtr, Xva = X_all.iloc[tr_idx], X_all.iloc[va_idx]
    ytr, yva = y_all.iloc[tr_idx], y_all.iloc[va_idx]

    spw_fold = (ytr == 0).sum() / (ytr == 1).sum()
    m = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        scale_pos_weight=spw_fold, eval_metric='aucpr',
        early_stopping_rounds=30, random_state=42, use_label_encoder=False
    )
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    proba = m.predict_proba(Xva)[:, 1]
    preds = (proba >= BEST_THRESHOLD).astype(int)
    cv_f1.append(f1_score(yva, preds, zero_division=0))
    cv_recall.append(recall_score(yva, preds, zero_division=0))
    print(f'Fold {fold+1}: F1={cv_f1[-1]:.3f}  Recall={cv_recall[-1]:.3f}')

print(f'\nMean F1:     {np.mean(cv_f1):.3f} ± {np.std(cv_f1):.3f}')
print(f'Mean Recall: {np.mean(cv_recall):.3f} ± {np.std(cv_recall):.3f}')

## Cell 14 — Retrain final model on all labeled data

In [ ]:
spw_final = (y_all == 0).sum() / (y_all == 1).sum()

final_model = xgb.XGBClassifier(
    n_estimators=xgb_model.best_iteration + 1,  # use best n from val tuning
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=spw_final,
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False
)

final_model.fit(X_all, y_all, verbose=False)
print('Final model trained on all labeled data.')

## Cell 15 — Generate test predictions & submission file

In [ ]:
# Test predictions
test_xgb_proba = final_model.predict_proba(X_test)[:, 1]
test_rule_flags, test_rule_reasons = apply_rules(test_df, X_test)

test_preds, test_scores = ensemble_predict(test_rule_flags, test_xgb_proba)

submission = pd.DataFrame({
    'call_id':          test_df['call_id'],
    'predicted_ticket': test_preds.astype(bool)
})

print(f'Test predictions: {submission["predicted_ticket"].sum()} flagged out of {len(submission)}')
print(f'Flag rate: {submission["predicted_ticket"].mean():.1%}')
print('\nFirst few rows:')
print(submission.head(10))

submission.to_csv('submission.csv', index=False)
print('\nSaved to submission.csv')

## Cell 16 — Download submission

In [ ]:
from google.colab import files
files.download('submission.csv')

---
## BONUS — Ticket Category Prediction

In [ ]:
# Build category labels for train+val (only for flagged calls)
cat_cols = [
    'ticket_cat_audio_issue',
    'ticket_cat_elevenlabs',
    'ticket_cat_openai',
    'ticket_cat_supabase',
    'ticket_cat_scheduler_aws',
    'ticket_cat_other'
]

def get_ticket_category(row):
    """
    Return the primary ticket category for a row.
    Returns 'no_issue' for non-ticket calls.
    """
    for col in cat_cols:
        if col in row.index and str(row.get(col, '')).lower() == 'issue':
            return col.replace('ticket_cat_', '')
    if row.get('has_ticket', False):
        return 'other'
    return 'no_issue'

all_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)
all_labels = all_df.apply(get_ticket_category, axis=1)

print('Category distribution:')
print(all_labels.value_counts())

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.multiclass import OneVsRestClassifier

# Only train on calls that have a ticket
ticket_mask = all_labels != 'no_issue'
X_cat = X_all[ticket_mask.values]
y_cat_raw = all_labels[ticket_mask.values]

le = LabelEncoder()
y_cat = le.fit_transform(y_cat_raw)

print(f'Category training set: {X_cat.shape}, Classes: {le.classes_}')

cat_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    objective='multi:softprob',
    num_class=len(le.classes_),
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)
cat_model.fit(X_cat, y_cat, verbose=False)
print('Category model trained.')

In [ ]:
# Add category predictions to submission
flagged_mask = test_preds == 1
submission['ticket_category'] = 'no_issue'

if flagged_mask.sum() > 0:
    X_test_flagged = X_test[flagged_mask]
    cat_preds = cat_model.predict(X_test_flagged)
    cat_labels = le.inverse_transform(cat_preds)
    submission.loc[flagged_mask, 'ticket_category'] = cat_labels

print('Submission with categories:')
print(submission[submission['predicted_ticket']].head(15))

submission.to_csv('submission_with_categories.csv', index=False)
print('\nSaved to submission_with_categories.csv')

In [ ]:
from google.colab import files
files.download('submission_with_categories.csv')